# ReAct Prompting — Exemplo real com ferramentas Python e Ollama

Este notebook constrói um exemplo didático, executável e realista de **ReAct Prompting**.

ReAct combina duas ideias:

- **Reasoning**: a LLM decide qual é o próximo passo, usando um plano curto e operacional.
- **Acting**: o sistema executa uma ferramenta real e devolve o resultado para a LLM.

A diferença central entre simular ReAct e executar ReAct de verdade é a origem da `Observation`.

Em uma simulação ruim, a própria LLM escreve `Thought`, `Action`, `Observation` e `Finish` em uma única resposta. Nesse caso, a observação pode ter sido inventada.

Neste notebook, o ciclo é real:

1. a LLM escolhe uma ação;
2. o código Python interpreta a ação;
3. o código executa uma ferramenta local;
4. a ferramenta gera a `Observation`;
5. a `Observation` volta ao histórico;
6. a LLM decide a próxima etapa;
7. o ciclo continua até `Finish` ou até o limite de rodadas.

A estrutura usada é:

`Thought → Action → Observation → Thought → Action → Observation → Finish`

Este exemplo usa **Ollama** com três modos: **`gemma3:4b`** local/offline, **`gemma4:31b-cloud`** via proxy local com `ollama signin`, ou **`gemma4:31b`** direto em `https://ollama.com` com `OLLAMA_API_KEY`. A chave é lida de variável de ambiente ou prompt seguro, nunca gravada no notebook.

Referências oficiais sobre modelos cloud e planos: <https://docs.ollama.com/cloud>, <https://ollama.com/cloud> e <https://ollama.com/library/gemma4:31b>.

O cenário é uma **Receita Estadual fictícia**. Os dados, regras, classificações e fundamentos são inteiramente fictícios e servem apenas para fins didáticos.

In [ ]:
%pip install -q langchain langchain-ollama pydantic rich requests

In [8]:
import getpass
import os
import requests
from pathlib import Path

# Escolha do modo para a aula.
# Opções:
# - "cloud_direto_api_key": acessa https://ollama.com com OLLAMA_API_KEY em header seguro.
# - "cloud_proxy_signin": usa o Ollama local como proxy; exige `ollama signin` no Windows.
# - "local": mantém execução local/offline com Gemma 3 4B.
MODO_OLLAMA = "cloud_proxy_signin"

OLLAMA_MODEL_LOCAL = "gemma3:4b"
OLLAMA_MODEL_CLOUD_PROXY = "gemma4:31b-cloud"
OLLAMA_MODEL_CLOUD_API = "gemma4:31b"
OLLAMA_API_KEY_ENV = "OLLAMA_API_KEY"

OLLAMA_API_KEY = None
OLLAMA_CLIENT_KWARGS = None


def obter_segredo_ollama_api_key() -> str | None:
    """Obtém a API key sem exibir nem gravar o valor no notebook."""
    valor = os.environ.get(OLLAMA_API_KEY_ENV)
    if valor:
        print(f"{OLLAMA_API_KEY_ENV} carregada de variável de ambiente. Valor não exibido.")
        return valor

    if MODO_OLLAMA != "cloud_direto_api_key":
        return None

    try:
        valor = getpass.getpass(
            f"Informe {OLLAMA_API_KEY_ENV} para Ollama Cloud direto. "
            "A chave não será exibida nem gravada no notebook: "
        ).strip()
    except Exception as erro:
        print(f"Não foi possível solicitar a chave de forma interativa: {erro}")
        return None

    if not valor:
        print(f"{OLLAMA_API_KEY_ENV} não informada. O notebook tentará fallback local/proxy.")
        return None

    print(f"{OLLAMA_API_KEY_ENV} recebida por prompt seguro. Valor não exibido nem persistido.")
    return valor


def obter_gateway_wsl() -> str | None:
    """Tenta descobrir o gateway do WSL a partir do resolv.conf."""
    resolv_conf = Path("/etc/resolv.conf")
    if not resolv_conf.exists():
        return None

    for linha in resolv_conf.read_text(encoding="utf-8", errors="ignore").splitlines():
        partes = linha.strip().split()
        if len(partes) == 2 and partes[0] == "nameserver":
            return partes[1]
    return None


candidatos_base_local = [
    "http://localhost:11434",
    "http://127.0.0.1:11434",
    "http://host.docker.internal:11434",
]

gateway_wsl = obter_gateway_wsl()
if gateway_wsl:
    candidatos_base_local.append(f"http://{gateway_wsl}:11434")

# Remove duplicatas preservando ordem.
candidatos_base_local = list(dict.fromkeys(candidatos_base_local))

OLLAMA_BASE_URL_LOCAL = None
dados_modelos = None

for base_url in candidatos_base_local:
    tags_url = f"{base_url.rstrip('/')}/api/tags"
    try:
        resposta = requests.get(tags_url, timeout=5)
        resposta.raise_for_status()
        OLLAMA_BASE_URL_LOCAL = base_url.rstrip("/")
        dados_modelos = resposta.json()
        print(f"Ollama local está rodando em: {OLLAMA_BASE_URL_LOCAL}")
        break
    except Exception as erro:
        print(f"Não foi possível acessar {tags_url}: {erro}")

modelos_locais = []
if dados_modelos:
    modelos_locais = [m.get("name") for m in dados_modelos.get("models", [])]
    print("\nModelos disponíveis no Ollama local:")
    for modelo in modelos_locais:
        print("-", modelo)


def configurar_modelo_local() -> tuple[str, str, dict | None]:
    """Configura fallback local/offline com o Ollama instalado."""
    base_url = OLLAMA_BASE_URL_LOCAL or "http://localhost:11434"
    if not OLLAMA_BASE_URL_LOCAL:
        print("\nOllama local não parece estar acessível a partir deste ambiente.")
        print("Se o Ollama estiver no Windows e o notebook rodar no WSL, tente host.docker.internal.")
        print("No Windows, inicie o Ollama e garanta que o modelo local exista:")
        print(f"ollama pull {OLLAMA_MODEL_LOCAL}")
        print("Se necessário, exponha o serviço com OLLAMA_HOST=0.0.0.0:11434 e reinicie o Ollama.")
    elif OLLAMA_MODEL_LOCAL not in modelos_locais:
        print(f"\nModelo local não encontrado: {OLLAMA_MODEL_LOCAL}")
        print(f"Execute no terminal: ollama pull {OLLAMA_MODEL_LOCAL}")
    return base_url, OLLAMA_MODEL_LOCAL, None


if MODO_OLLAMA == "cloud_direto_api_key":
    OLLAMA_API_KEY = obter_segredo_ollama_api_key()
    if OLLAMA_API_KEY:
        OLLAMA_BASE_URL = "https://ollama.com"
        OLLAMA_MODEL = OLLAMA_MODEL_CLOUD_API
        OLLAMA_CLIENT_KWARGS = {
            "headers": {"Authorization": f"Bearer {OLLAMA_API_KEY}"}
        }
        print("\nModo cloud direto ativado via API key em memória.")
        print(f"Modelo remoto selecionado: {OLLAMA_MODEL}")
    else:
        print("\nSem API key disponível. Usando fallback local.")
        OLLAMA_BASE_URL, OLLAMA_MODEL, OLLAMA_CLIENT_KWARGS = configurar_modelo_local()
elif MODO_OLLAMA == "cloud_proxy_signin":
    OLLAMA_BASE_URL = OLLAMA_BASE_URL_LOCAL or "http://localhost:11434"
    OLLAMA_MODEL = OLLAMA_MODEL_CLOUD_PROXY
    OLLAMA_CLIENT_KWARGS = None
    print(f"\nModo cloud via proxy local ativado: {OLLAMA_MODEL}")
    print("Este modo exige `ollama signin` no Windows.")
    print(f"Se necessário, execute: ollama pull {OLLAMA_MODEL_CLOUD_PROXY}")
elif MODO_OLLAMA == "local":
    OLLAMA_BASE_URL, OLLAMA_MODEL, OLLAMA_CLIENT_KWARGS = configurar_modelo_local()
    print(f"\nModo local ativado: {OLLAMA_MODEL}")
else:
    raise ValueError(
        "MODO_OLLAMA deve ser 'cloud_direto_api_key', 'cloud_proxy_signin' ou 'local'."
    )

from langchain_ollama import ChatOllama


def criar_llm(
    modelo: str,
    base_url: str | None = None,
    client_kwargs: dict | None = None,
) -> ChatOllama:
    """Cria uma instância ChatOllama sem imprimir segredos."""
    return ChatOllama(
        model=modelo,
        base_url=base_url or OLLAMA_BASE_URL,
        temperature=0,
        client_kwargs=client_kwargs or {},
    )


llm = criar_llm(OLLAMA_MODEL, OLLAMA_BASE_URL, OLLAMA_CLIENT_KWARGS)


def ativar_fallback_local():
    """Troca a LLM ativa para o modelo local."""
    global llm, OLLAMA_BASE_URL, OLLAMA_MODEL, OLLAMA_CLIENT_KWARGS
    OLLAMA_BASE_URL, OLLAMA_MODEL, OLLAMA_CLIENT_KWARGS = configurar_modelo_local()
    llm = criar_llm(OLLAMA_MODEL, OLLAMA_BASE_URL, OLLAMA_CLIENT_KWARGS)


def invocar_llm(mensagens):
    """Invoca a LLM e faz fallback para o modelo local em caso de erro."""
    try:
        return llm.invoke(mensagens)
    except Exception as erro:
        if OLLAMA_MODEL != OLLAMA_MODEL_LOCAL:
            print("\nFalha ao usar o modo cloud do Ollama.")
            print(f"Erro recebido: {erro}")
            print(f"Fazendo fallback para o modelo local: {OLLAMA_MODEL_LOCAL}")
            ativar_fallback_local()
            return llm.invoke(mensagens)
        raise


print(f"\nModelo selecionado para esta execução: {OLLAMA_MODEL}")
print(f"Base URL selecionada: {OLLAMA_BASE_URL}")
print("Segredos: nenhum valor de chave foi impresso ou gravado no notebook.")

Não foi possível acessar http://localhost:11434/api/tags: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
Não foi possível acessar http://127.0.0.1:11434/api/tags: HTTPConnectionPool(host='127.0.0.1', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))
Ollama local está rodando em: http://host.docker.internal:11434

Modelos disponíveis no Ollama local:
- gemma4:31b-cloud
- phi4-mini:latest
- gemma3:4b

Modo cloud via proxy local ativado: gemma4:31b-cloud
Este modo exige `ollama signin` no Windows.
Se necessário, execute: ollama pull gemma4:31b-cloud

Modelo selecionado para esta execução: gemma4:31b-cloud
Base URL selecionada: http://host.docker.int

In [9]:
pedido_usuario = "Quero os valores de operações fiscais por CNPJ, NCM e unidade fiscal no período de janeiro a março de 2026."

pedido_usuario

'Quero os valores de operações fiscais por CNPJ, NCM e unidade fiscal no período de janeiro a março de 2026.'

In [10]:
import re
import unicodedata


def normalizar_texto(texto: str) -> str:
    """Normaliza texto para comparação simples e determinística."""
    texto = texto.lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(caractere for caractere in texto if not unicodedata.combining(caractere))
    return texto


def normalizar_campos(campos: list[str]) -> set[str]:
    """Normaliza nomes de campos para regras locais."""
    return {normalizar_texto(str(campo)).strip() for campo in campos}


def classificar_pedido(texto: str) -> dict:
    """Classifica campos e indícios presentes no pedido do cidadão."""
    texto_norm = normalizar_texto(texto)

    contem_cnpj = bool(re.search(r"\bcnpj\b|\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}", texto_norm))
    contem_ncm = "ncm" in texto_norm
    contem_periodo = bool(
        re.search(
            r"\bperiodo\b|\bmes\b|\bmeses\b|janeiro|fevereiro|marco|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro|\b20\d{2}\b",
            texto_norm,
        )
    )
    contem_valores = bool(re.search(r"\bvalor\b|\bvalores\b|montante|operacoes fiscais|operacoes", texto_norm))
    contem_unidade_fiscal = "unidade fiscal" in texto_norm
    contem_destinatarios = "destinatario" in texto_norm or "destinatarios" in texto_norm
    contem_remetentes = "remetente" in texto_norm or "remetentes" in texto_norm
    contem_empresas = "empresa" in texto_norm or "empresas" in texto_norm
    pede_lista = "lista" in texto_norm or "relacao" in texto_norm
    menciona_agregado = "agregado" in texto_norm or "total" in texto_norm or "sem identificacao" in texto_norm
    recusa_dados_agregados = bool(
        re.search(r"\bnao aceito\b.*\bagregad|\bnao quero\b.*\bagregad|\bpreciso\b.*\bindividualizad", texto_norm)
    )

    campos_detectados = []
    if contem_cnpj:
        campos_detectados.append("CNPJ")
    if contem_ncm:
        campos_detectados.append("NCM")
    if contem_unidade_fiscal:
        campos_detectados.append("unidade fiscal")
    if contem_periodo:
        campos_detectados.append("período")
    if contem_valores:
        campos_detectados.append("valores de operações")
    if contem_destinatarios:
        campos_detectados.append("destinatários")
    if contem_remetentes:
        campos_detectados.append("remetentes")
    if contem_empresas:
        campos_detectados.append("empresas")

    if contem_destinatarios or contem_remetentes or contem_empresas or (contem_cnpj and contem_valores):
        risco_sigilo_preliminar = "alto"
    elif contem_cnpj:
        risco_sigilo_preliminar = "medio"
    else:
        risco_sigilo_preliminar = "baixo"

    if contem_cnpj or contem_destinatarios or contem_remetentes or contem_empresas or pede_lista:
        tipo_pedido = "individualizado"
    elif menciona_agregado:
        tipo_pedido = "agregado"
    else:
        tipo_pedido = "indeterminado"

    return {
        "contem_cnpj": contem_cnpj,
        "contem_ncm": contem_ncm,
        "contem_periodo": contem_periodo,
        "contem_valores": contem_valores,
        "contem_unidade_fiscal": contem_unidade_fiscal,
        "contem_destinatarios": contem_destinatarios,
        "contem_remetentes": contem_remetentes,
        "tipo_pedido": tipo_pedido,
        "campos_detectados": campos_detectados,
        "risco_sigilo_preliminar": risco_sigilo_preliminar,
        "parece_pedir_dado_agregado": tipo_pedido == "agregado",
        "parece_pedir_dado_individualizado": tipo_pedido == "individualizado",
        "recusa_dados_agregados": recusa_dados_agregados,
    }


def verificar_sigilo_fiscal(campos: list[str] | dict) -> dict:
    """Avalia risco fictício de sigilo fiscal a partir dos campos solicitados."""
    if isinstance(campos, dict):
        campos_base = list(campos.get("campos_detectados") or campos.get("campos") or [])
        for chave, campo in [
            ("contem_cnpj", "CNPJ"),
            ("contem_valores", "valores de operações"),
            ("contem_destinatarios", "destinatários"),
            ("contem_remetentes", "remetentes"),
            ("parece_pedir_dado_individualizado", "dados individualizados por contribuinte"),
        ]:
            if campos.get(chave) and campo not in campos_base:
                campos_base.append(campo)
        campos = campos_base

    campos_norm = normalizar_campos(campos)

    tem_cnpj = "cnpj" in campos_norm
    tem_valores = "valores de operacoes" in campos_norm or "valor total agregado" in campos_norm
    tem_ncm = "ncm" in campos_norm
    tem_periodo = "periodo" in campos_norm or "mes" in campos_norm
    tem_unidade = "unidade fiscal" in campos_norm
    tem_destinatarios = "destinatarios" in campos_norm
    tem_remetentes = "remetentes" in campos_norm
    tem_empresas = "empresas" in campos_norm
    tem_individualizado = "dados individualizados por contribuinte" in campos_norm or "registros individualizados" in campos_norm

    campos_sensiveis = []
    if tem_cnpj:
        campos_sensiveis.append("CNPJ")
    if tem_valores and (tem_cnpj or tem_individualizado):
        campos_sensiveis.append("valores de operações")
    if tem_destinatarios:
        campos_sensiveis.append("destinatários")
    if tem_remetentes:
        campos_sensiveis.append("remetentes")
    if tem_empresas:
        campos_sensiveis.append("empresas")
    if tem_individualizado:
        campos_sensiveis.append("dados individualizados por contribuinte")

    if (tem_cnpj and tem_valores) or tem_destinatarios or tem_remetentes or tem_empresas or tem_individualizado:
        risco = "alto"
        justificativa = "A combinação solicitada permite identificação individualizada de contribuinte ou de participantes da operação."
    elif tem_cnpj or (tem_unidade and tem_valores and not tem_ncm):
        risco = "medio"
        justificativa = "Há elementos que podem aproximar a análise de contribuintes específicos, dependendo do recorte."
    elif tem_ncm and tem_periodo and not tem_cnpj:
        risco = "baixo"
        justificativa = "O recorte por NCM e período, sem CNPJ, tende a permitir divulgação agregada."
    else:
        risco = "baixo"
        justificativa = "Não foram detectados campos fortemente individualizantes no recorte informado."

    return {
        "risco_sigilo": risco,
        "campos_sensiveis": campos_sensiveis,
        "justificativa": justificativa,
    }


def consultar_disponibilidade_relatorio(campos: list[str]) -> dict:
    """Simula disponibilidade de relatório estruturado para fornecimento externo."""
    campos_norm = normalizar_campos(campos)

    tem_cnpj = "cnpj" in campos_norm
    tem_ncm = "ncm" in campos_norm
    tem_periodo = "periodo" in campos_norm or "mes" in campos_norm
    tem_unidade = "unidade fiscal" in campos_norm
    tem_valores = "valores de operacoes" in campos_norm or "valor total agregado" in campos_norm
    tem_destinatarios = "destinatarios" in campos_norm
    tem_remetentes = "remetentes" in campos_norm
    tem_faixa = "faixa de valor" in campos_norm or "faixas de valor" in campos_norm

    if tem_destinatarios or tem_remetentes:
        return {
            "relatorio_disponivel": False,
            "tipo_relatorio": None,
            "campos_possiveis": ["NCM", "unidade fiscal", "período", "valor total agregado"],
            "restricoes": ["sem destinatários", "sem remetentes", "sem identificação individualizada de contribuinte"],
            "justificativa": "Relatórios com identificação de destinatários ou remetentes não estão disponíveis para fornecimento externo neste exemplo fictício.",
        }

    if tem_cnpj and tem_valores:
        return {
            "relatorio_disponivel": False,
            "tipo_relatorio": None,
            "campos_possiveis": ["NCM", "unidade fiscal", "período", "valor total agregado"],
            "restricoes": ["sem CNPJ", "sem identificação individualizada de contribuinte"],
            "justificativa": "Relatório por CNPJ com valores individualizados não pode ser fornecido diretamente neste exemplo fictício.",
        }

    if tem_faixa:
        return {
            "relatorio_disponivel": False,
            "tipo_relatorio": None,
            "campos_possiveis": [],
            "restricoes": ["faixas de valor não existem como relatório estruturado"],
            "justificativa": "Não há relatório estruturado por faixas de valor neste exemplo fictício.",
        }

    if tem_ncm and tem_periodo and (tem_unidade or tem_valores):
        return {
            "relatorio_disponivel": True,
            "tipo_relatorio": "agregado",
            "campos_possiveis": ["NCM", "unidade fiscal", "período", "valor total agregado"],
            "restricoes": ["sem CNPJ", "sem identificação individualizada de contribuinte"],
        }

    return {
        "relatorio_disponivel": True,
        "tipo_relatorio": "agregado_basico",
        "campos_possiveis": ["período", "valor total agregado"],
        "restricoes": ["sem CNPJ", "sem identificação individualizada de contribuinte"],
    }


def propor_alternativa_agregada(pedido: str) -> dict:
    """Propõe alternativa agregada para atendimento parcial do pedido."""
    pedido_norm = normalizar_texto(pedido)
    recusa_dados_agregados = bool(
        re.search(r"\bnao aceito\b.*\bagregad|\bnao quero\b.*\bagregad|\bpreciso\b.*\bindividualizad", pedido_norm)
    )

    if recusa_dados_agregados:
        return {
            "alternativa_disponivel": False,
            "alternativa": None,
            "campos_sugeridos": [],
            "campos_excluidos": ["CNPJ", "empresas", "remetentes", "destinatários", "dados individualizados por contribuinte"],
            "observacao": "O pedido exige registros individualizados e recusa dados agregados; neste exemplo fictício, a decisão adequada é negativa.",
        }

    return {
        "alternativa_disponivel": True,
        "alternativa": "Fornecer relatório agregado por NCM, unidade fiscal e período, sem CNPJ.",
        "campos_sugeridos": ["NCM", "unidade fiscal", "mês", "valor total agregado"],
        "campos_excluidos": ["CNPJ", "dados individualizados por contribuinte"],
        "observacao": "A alternativa preserva o recorte analítico principal sem expor contribuinte individualizado.",
    }


def gerar_minuta_resposta(decisao: str, fundamentos: list[str] | str, alternativa: dict | str | None = None) -> str:
    """Gera minuta administrativa curta para o caso analisado."""
    rotulos_decisao = {
        "atendimento_integral": "atendimento integral",
        "atendimento_parcial": "atendimento parcial",
        "negativa": "negativa de fornecimento",
        "esclarecimento": "necessidade de esclarecimento",
    }
    decisao_texto = rotulos_decisao.get(decisao, decisao)

    if isinstance(fundamentos, str):
        fundamentos_iteraveis = [fundamentos]
    elif fundamentos is None:
        fundamentos_iteraveis = []
    else:
        fundamentos_iteraveis = fundamentos

    fundamentos_limpos = [str(item).strip() for item in fundamentos_iteraveis if str(item).strip()]
    if not fundamentos_limpos:
        fundamentos_limpos = ["A análise foi feita com base nas regras fictícias deste exemplo didático."]

    linhas = [
        "Minuta de resposta administrativa - exemplo fictício e didático.",
        f"Decisão sugerida: {decisao_texto}.",
        "Fundamentos da triagem:",
    ]

    for indice, fundamento in enumerate(fundamentos_limpos, start=1):
        linhas.append(f"{indice}. {fundamento}")

    alternativa_dict = alternativa if isinstance(alternativa, dict) else {}
    alternativa_texto = ""
    if isinstance(alternativa, str):
        alternativa_texto = alternativa.strip()
    elif alternativa_dict:
        alternativa_texto = str(alternativa_dict.get("alternativa", "Fornecer dados em formato agregado, sem identificação individualizada.")).strip()

    if alternativa_texto or alternativa_dict:
        linhas.append("Alternativa de atendimento sugerida:")
        linhas.append(alternativa_texto or "Fornecer dados em formato agregado, sem identificação individualizada.")
        campos_sugeridos = alternativa_dict.get("campos_sugeridos") or []
        campos_excluidos = alternativa_dict.get("campos_excluidos") or []
        if campos_sugeridos:
            linhas.append("Campos sugeridos: " + ", ".join(map(str, campos_sugeridos)) + ".")
        if campos_excluidos:
            linhas.append("Campos excluídos: " + ", ".join(map(str, campos_excluidos)) + ".")

    linhas.append("Esta minuta não cita legislação real como fundamento definitivo e deve ser revisada por responsável competente.")
    return "\n".join(linhas)

In [11]:
TOOLS = {
    "classificar_pedido": classificar_pedido,
    "verificar_sigilo_fiscal": verificar_sigilo_fiscal,
    "consultar_disponibilidade_relatorio": consultar_disponibilidade_relatorio,
    "propor_alternativa_agregada": propor_alternativa_agregada,
    "gerar_minuta_resposta": gerar_minuta_resposta,
}

list(TOOLS.keys())

['classificar_pedido',
 'verificar_sigilo_fiscal',
 'consultar_disponibilidade_relatorio',
 'propor_alternativa_agregada',
 'gerar_minuta_resposta']

In [12]:
SYSTEM_PROMPT = """
Você é um agente de apoio à triagem de pedidos de acesso à informação em uma Receita Estadual fictícia.

Objetivo:
Analisar o pedido do cidadão e decidir se deve ser atendido integralmente, parcialmente, negado ou se precisa de esclarecimento.

Regras:
- Use apenas as ferramentas listadas.
- Faça uma única ação por rodada.
- Não invente observações.
- A Observation será fornecida pelo sistema.
- Não informe dados individualizados protegidos por sigilo fiscal.
- Priorize atendimento parcial quando for possível fornecer dados agregados.
- Se o pedido exigir registros individualizados e recusar dados agregados, não proponha atendimento parcial; gere minuta com decisão "negativa".
- Se uma Observation indicar "alternativa_disponivel": false, use decisão "negativa" ao chamar gerar_minuta_resposta.
- Antes de finalizar, use gerar_minuta_resposta para obter uma minuta administrativa.
- Quando tiver informação suficiente e uma minuta observada, responda com Finish.
- Finish não é ferramenta: nunca use "action": "finish".
- Para finalizar, use somente a chave top-level "finish", conforme o formato obrigatório de saída para finalizar.
- Não exponha raciocínio interno longo.
- Use o campo "thought" com no máximo uma frase objetiva.
- Não use markdown.
- Não coloque ```json.
- Responda somente com JSON válido.
- Não escreva texto antes ou depois do JSON.
- Se receber erro de ferramenta ou erro de formato, corrija na rodada seguinte.
- Ao chamar ferramentas com campos, use os campos retornados pelas observações anteriores.
- Ao chamar verificar_sigilo_fiscal, passe a lista completa de campos_detectados retornada por classificar_pedido.
- Se houver CNPJ com valores, destinatários, remetentes, empresas ou dados individualizados, considere risco de sigilo "alto".

Ferramentas disponíveis:
- classificar_pedido(texto)
- verificar_sigilo_fiscal(campos)
- consultar_disponibilidade_relatorio(campos)
- propor_alternativa_agregada(pedido)
- gerar_minuta_resposta(decisao, fundamentos, alternativa)
  - decisao: string, por exemplo "atendimento_parcial".
  - fundamentos: lista de strings; não envie uma string única.
  - alternativa: objeto JSON com alternativa, campos_sugeridos e campos_excluidos; use null se não houver alternativa.

Formato obrigatório de saída para ação:

{
  "thought": "frase curta sobre a próxima etapa",
  "action": "nome_da_ferramenta",
  "args": {
    "parametro": "valor"
  }
}

Formato obrigatório de saída para finalizar:

{
  "thought": "frase curta indicando conclusão",
  "finish": {
    "decisao": "atendimento_integral | atendimento_parcial | negativa | esclarecimento",
    "resposta_final": "texto da resposta final ao gestor"
  }
}
""".strip()

print(SYSTEM_PROMPT[:800] + "...")

Você é um agente de apoio à triagem de pedidos de acesso à informação em uma Receita Estadual fictícia.

Objetivo:
Analisar o pedido do cidadão e decidir se deve ser atendido integralmente, parcialmente, negado ou se precisa de esclarecimento.

Regras:
- Use apenas as ferramentas listadas.
- Faça uma única ação por rodada.
- Não invente observações.
- A Observation será fornecida pelo sistema.
- Não informe dados individualizados protegidos por sigilo fiscal.
- Priorize atendimento parcial quando for possível fornecer dados agregados.
- Se o pedido exigir registros individualizados e recusar dados agregados, não proponha atendimento parcial; gere minuta com decisão "negativa".
- Se uma Observation indicar "alternativa_disponivel": false, use decisão "negativa" ao chamar gerar_minuta_respos...


In [13]:
import json
import re


def remover_cercas_markdown(texto: str) -> str:
    """Remove cercas comuns de Markdown sem alterar o conteúdo JSON interno."""
    texto = texto.strip()
    texto = re.sub(r"^```(?:json)?\s*", "", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s*```$", "", texto)
    return texto.strip()


def extrair_primeiro_objeto_json(texto: str) -> str | None:
    """Extrai o primeiro objeto JSON balanceado encontrado no texto."""
    inicio = texto.find("{")
    while inicio != -1:
        profundidade = 0
        em_string = False
        escape = False

        for posicao in range(inicio, len(texto)):
            caractere = texto[posicao]

            if em_string:
                if escape:
                    escape = False
                elif caractere == "\\":
                    escape = True
                elif caractere == '"':
                    em_string = False
                continue

            if caractere == '"':
                em_string = True
            elif caractere == "{":
                profundidade += 1
            elif caractere == "}":
                profundidade -= 1
                if profundidade == 0:
                    return texto[inicio : posicao + 1]

        inicio = texto.find("{", inicio + 1)

    return None


def parse_llm_json(resposta: str) -> dict:
    """Converte a resposta da LLM para dict, tolerando cercas Markdown e texto extra."""
    conteudo = remover_cercas_markdown(str(resposta))

    try:
        objeto = json.loads(conteudo)
        if isinstance(objeto, dict):
            return objeto
    except json.JSONDecodeError:
        pass

    candidato = extrair_primeiro_objeto_json(conteudo)
    if candidato:
        try:
            objeto = json.loads(candidato)
            if isinstance(objeto, dict):
                return objeto
        except json.JSONDecodeError as erro:
            return {
                "erro_parse_json": True,
                "mensagem": str(erro),
                "conteudo_original": resposta,
            }

    return {
        "erro_parse_json": True,
        "conteudo_original": resposta,
    }


def extrair_finish(decisao: dict) -> dict | None:
    """Aceita tanto o formato correto de Finish quanto variações comuns da LLM."""
    if not isinstance(decisao, dict):
        return None

    finish = decisao.get("finish")
    if isinstance(finish, dict):
        return finish

    action = str(decisao.get("action", "")).strip().lower()
    args = decisao.get("args")
    if action == "finish" and isinstance(args, dict):
        finish = args.get("finish", args)
        if isinstance(finish, dict) and {"decisao", "resposta_final"}.issubset(finish):
            return finish

    return None


In [14]:
def executar_ferramenta(action: str, args: dict) -> dict | str:
    """Executa uma ferramenta registrada de forma segura, sem eval ou código arbitrário."""
    if action not in TOOLS:
        return {
            "erro": "ferramenta_inexistente",
            "mensagem": f"A ferramenta {action} não existe.",
            "ferramentas_disponiveis": list(TOOLS.keys()),
        }

    if not isinstance(args, dict):
        return {
            "erro": "argumentos_invalidos",
            "mensagem": "Os argumentos devem ser enviados como objeto JSON.",
            "action": action,
            "args_recebidos": args,
        }

    try:
        return TOOLS[action](**args)
    except TypeError as erro:
        return {
            "erro": "argumentos_invalidos",
            "mensagem": str(erro),
            "action": action,
            "args_recebidos": args,
        }
    except Exception as erro:
        return {
            "erro": "erro_na_ferramenta",
            "mensagem": str(erro),
            "action": action,
        }

In [15]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from rich.console import Console
from rich.panel import Panel

console = Console()

historico = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=f"Pedido do cidadão: {pedido_usuario}"),
]

MAX_RODADAS = 8
MAX_TENTATIVAS_PARSE = 2
tentativas_parse = 0
resultado_final = None

for rodada in range(1, MAX_RODADAS + 1):
    console.rule(f"Rodada {rodada}")

    resposta = invocar_llm(historico)
    conteudo = str(resposta.content).strip()

    console.print(Panel(conteudo, title="Resposta bruta da LLM", expand=False))

    decisao = parse_llm_json(conteudo)

    if decisao.get("erro_parse_json"):
        tentativas_parse += 1
        historico.append(AIMessage(content=conteudo))
        console.print("Erro de parse JSON. Solicitando correção de formato.", style="bold red")

        if tentativas_parse > MAX_TENTATIVAS_PARSE:
            resultado_final = {
                "erro": "limite_tentativas_parse",
                "mensagem": "A LLM não retornou JSON válido dentro do limite de tentativas.",
            }
            break

        historico.append(
            HumanMessage(
                content=(
                    "Observation: Sua resposta não estava em JSON válido. "
                    "Responda novamente usando apenas JSON válido, sem markdown, no formato especificado."
                )
            )
        )
        continue

    tentativas_parse = 0
    thought = decisao.get("thought", "")
    console.print(f"Thought: {thought}")

    finish = extrair_finish(decisao)
    if finish is not None:
        historico.append(AIMessage(content=conteudo))
        resultado_final = finish
        console.print(Panel(json.dumps(resultado_final, ensure_ascii=False, indent=2), title="Finish", expand=False))
        break

    action = decisao.get("action")
    args = decisao.get("args", {})
    console.print(f"Action: {action}")
    console.print(f"Args: {json.dumps(args, ensure_ascii=False)}")

    observation = executar_ferramenta(action, args)
    console.print(Panel(json.dumps(observation, ensure_ascii=False, indent=2), title="Observation gerada pelo Python", expand=False))

    historico.append(AIMessage(content=conteudo))
    historico.append(HumanMessage(content="Observation: " + json.dumps(observation, ensure_ascii=False)))
else:
    resultado_final = {
        "erro": "limite_rodadas",
        "mensagem": "Limite de rodadas atingido sem Finish.",
    }
    console.print("Limite de rodadas atingido sem Finish.", style="bold yellow")

resultado_final

──────────────────────────────────────────────────── Rodada 1 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "thought": "Classificando o pedido para identificar os campos solicitados.",                                  │
│   "action": "classificar_pedido",                                                                               │
│   "args": {                                                                                                     │
│     "texto": "Quero os valores de operações fiscais por CNPJ, NCM e unidade fiscal no período de janeiro a      │
│ março de 2026."                                                                                                 │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: Classificando o pedido para identificar os campos solicitados.

Action: classificar_pedido

Args: {"texto": "Quero os valores de operações fiscais por CNPJ, NCM e unidade fiscal no período de janeiro a março
de 2026."}

...

──────────────────────────────────────────────────── Rodada 6 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "thought": "Concluindo a triagem com a minuta de atendimento parcial gerada.",                                │
│   "finish": {                                                                                                   │
│     "decisao": "atendimento_parcial",                                                                           │
│     "resposta_final": "A triagem do pedido resultou em atendimento parcial. O fornecimento de dados             │
│ individualizados por CNPJ e valores de operações foi negado devido ao alto risco de quebra de sigilo fiscal.    │
│ Como alternativa, propõe-se a entrega de relatório agregado por NCM, unidade fiscal e período, excluindo-se a   │
│ identificação de contribuintes específicos para preservar a confidencialidade fiscal."                          │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: Concluindo a triagem com a minuta de atendimento parcial gerada.

╭──────────────────────────────────────────────────── Finish ─────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "decisao": "atendimento_parcial",                                                                             │
│   "resposta_final": "A triagem do pedido resultou em atendimento parcial. O fornecimento de dados               │
│ individualizados por CNPJ e valores de operações foi negado devido ao alto risco de quebra de sigilo fiscal.    │
│ Como alternativa, propõe-se a entrega de relatório agregado por NCM, unidade fiscal e período, excluindo-se a   │
│ identificação de contribuintes específicos para preservar a confidencialidade fiscal."                          │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'decisao': 'atendimento_parcial',
 'resposta_final': 'A triagem do pedido resultou em atendimento parcial. O fornecimento de dados individualizados por CNPJ e valores de operações foi negado devido ao alto risco de quebra de sigilo fiscal. Como alternativa, propõe-se a entrega de relatório agregado por NCM, unidade fiscal e período, excluindo-se a identificação de contribuintes específicos para preservar a confidencialidade fiscal.'}

## Execução esperada

Uma trajetória provável para o pedido principal é:

1. `classificar_pedido`: identifica CNPJ, NCM, unidade fiscal, período e valores de operações.
2. `verificar_sigilo_fiscal`: aponta alto risco fictício de sigilo porque há CNPJ combinado com valores.
3. `consultar_disponibilidade_relatorio`: informa que relatório individualizado por CNPJ e valores não pode ser fornecido diretamente.
4. `propor_alternativa_agregada`: sugere relatório agregado por NCM, unidade fiscal e período, sem CNPJ.
5. `gerar_minuta_resposta`: produz uma minuta administrativa curta.
6. `Finish`: finaliza com tendência a **atendimento parcial**.

A trajetória pode variar um pouco conforme a resposta do modelo, mas a lógica ReAct deve permanecer: a LLM escolhe uma ação, o Python executa a ferramenta, a observação volta para o histórico e a LLM decide a próxima etapa.

In [16]:
def executar_agente_react(pedido: str, max_rodadas: int = 8):
    """Executa o agente ReAct para um pedido e retorna decisão final, histórico e registros."""
    historico_local = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Pedido do cidadão: {pedido}"),
    ]
    registros = []
    tentativas_parse_local = 0

    for rodada in range(1, max_rodadas + 1):
        console.rule(f"Rodada {rodada}")

        resposta = invocar_llm(historico_local)
        conteudo = str(resposta.content).strip()
        decisao = parse_llm_json(conteudo)

        registro = {
            "rodada": rodada,
            "resposta_bruta": conteudo,
            "decisao_parseada": decisao,
        }
        registros.append(registro)

        console.print(Panel(conteudo, title="Resposta bruta da LLM", expand=False))

        if decisao.get("erro_parse_json"):
            tentativas_parse_local += 1
            historico_local.append(AIMessage(content=conteudo))
            console.print("Erro de parse JSON. Solicitando correção de formato.", style="bold red")

            if tentativas_parse_local > MAX_TENTATIVAS_PARSE:
                resultado = {
                    "erro": "limite_tentativas_parse",
                    "mensagem": "A LLM não retornou JSON válido dentro do limite de tentativas.",
                }
                return {"decisao_final": resultado, "historico": historico_local, "rodadas": registros}

            historico_local.append(
                HumanMessage(
                    content=(
                        "Observation: Sua resposta não estava em JSON válido. "
                        "Responda novamente usando apenas JSON válido, sem markdown, no formato especificado."
                    )
                )
            )
            continue

        tentativas_parse_local = 0
        thought = decisao.get("thought", "")
        console.print(f"Thought: {thought}")

        finish = extrair_finish(decisao)
        if finish is not None:
            historico_local.append(AIMessage(content=conteudo))
            resultado = finish
            registro["finish"] = resultado
            console.print(Panel(json.dumps(resultado, ensure_ascii=False, indent=2), title="Finish", expand=False))
            return {"decisao_final": resultado, "historico": historico_local, "rodadas": registros}

        action = decisao.get("action")
        args = decisao.get("args", {})
        console.print(f"Action: {action}")
        console.print(f"Args: {json.dumps(args, ensure_ascii=False)}")

        observation = executar_ferramenta(action, args)
        registro["action"] = action
        registro["args"] = args
        registro["observation"] = observation

        console.print(Panel(json.dumps(observation, ensure_ascii=False, indent=2), title="Observation gerada pelo Python", expand=False))

        historico_local.append(AIMessage(content=conteudo))
        historico_local.append(HumanMessage(content="Observation: " + json.dumps(observation, ensure_ascii=False)))

    resultado = {
        "erro": "limite_rodadas",
        "mensagem": "Limite de rodadas atingido sem Finish.",
    }
    console.print("Limite de rodadas atingido sem Finish.", style="bold yellow")
    return {"decisao_final": resultado, "historico": historico_local, "rodadas": registros}

In [17]:
pedido_1 = "Quero a lista dos CNPJ das empresas."

resultado_1 = executar_agente_react(pedido_1)
resultado_1["decisao_final"]

──────────────────────────────────────────────────── Rodada 1 ─────────────────────────────────────────────────────

╭──────────────────────────── Resposta bruta da LLM ─────────────────────────────╮
│ {                                                                              │
│   "thought": "Classificando o pedido para identificar os campos solicitados.", │
│   "action": "classificar_pedido",                                              │
│   "args": {                                                                    │
│     "texto": "Quero a lista dos CNPJ das empresas."                            │
│   }                                                                            │
│ }                                                                              │
╰────────────────────────────────────────────────────────────────────────────────╯

Thought: Classificando o pedido para identificar os campos solicitados.

Action: classificar_pedido

Args: {"texto": "Quero a lista dos CNPJ das empresas."}

...

──────────────────────────────────────────────────── Rodada 6 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "thought": "Concluindo a triagem com a decisão de atendimento parcial e a minuta gerada.",                    │
│   "finish": {                                                                                                   │
│     "decisao": "atendimento_parcial",                                                                           │
│     "resposta_final": "Minuta de resposta administrativa - exemplo fictício e didático.\nDecisão sugerida:      │
│ atendimento parcial.\nFundamentos da triagem:\n1. O pedido de lista de CNPJ envolve dados protegidos por sigilo │
│ fiscal\n2. A divulgação de dados individualizados de contribuintes é vedada por lei\n3. É possível fornecer     │
│ informações de forma agregada para preservar a privacidade dos contribuintes\nAlternativa de atendimento        │
│ sugerida:\nFornecer relatório agregado por NCM, unidade fiscal e período, sem CNPJ.\nCampos sugeridos: NCM,     │
│ unidade fiscal, mês, valor total agregado.\nCampos excluídos: CNPJ, dados individualizados por                  │
│ contribuinte.\nEsta minuta não cita legislação real como fundamento definitivo e deve ser revisada por          │
│ responsável competente."                                                                                        │
│   }                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: Concluindo a triagem com a decisão de atendimento parcial e a minuta gerada.

╭──────────────────────────────────────────────────── Finish ─────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "decisao": "atendimento_parcial",                                                                             │
│   "resposta_final": "Minuta de resposta administrativa - exemplo fictício e didático.\nDecisão sugerida:        │
│ atendimento parcial.\nFundamentos da triagem:\n1. O pedido de lista de CNPJ envolve dados protegidos por sigilo │
│ fiscal\n2. A divulgação de dados individualizados de contribuintes é vedada por lei\n3. É possível fornecer     │
│ informações de forma agregada para preservar a privacidade dos contribuintes\nAlternativa de atendimento        │
│ sugerida:\nFornecer relatório agregado por NCM, unidade fiscal e período, sem CNPJ.\nCampos sugeridos: NCM,     │
│ unidade fiscal, mês, valor total agregado.\nCampos excluídos: CNPJ, dados individualizados por                  │
│ contribuinte.\nEsta minuta não cita legislação real como fundamento definitivo e deve ser revisada por          │
│ responsável competente."                                                                                        │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'decisao': 'atendimento_parcial',
 'resposta_final': 'Minuta de resposta administrativa - exemplo fictício e didático.\nDecisão sugerida: atendimento parcial.\nFundamentos da triagem:\n1. O pedido de lista de CNPJ envolve dados protegidos por sigilo fiscal\n2. A divulgação de dados individualizados de contribuintes é vedada por lei\n3. É possível fornecer informações de forma agregada para preservar a privacidade dos contribuintes\nAlternativa de atendimento sugerida:\nFornecer relatório agregado por NCM, unidade fiscal e período, sem CNPJ.\nCampos sugeridos: NCM, unidade fiscal, mês, valor total agregado.\nCampos excluídos: CNPJ, dados individualizados por contribuinte.\nEsta minuta não cita legislação real como fundamento definitivo e deve ser revisada por responsável competente.'}

In [41]:
pedido_2 = "Solicito o total agregado por NCM e por mês, sem identificação dos contribuintes."

resultado_2 = executar_agente_react(pedido_2)
resultado_2["decisao_final"]

──────────────────────────────────────────────────── Rodada 1 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ ```json                                                                                                         │
│ {                                                                                                               │
│   "thought": "Preciso classificar o pedido para identificar os campos relevantes e avaliar o risco de sigilo    │
│ fiscal.",                                                                                                       │
│   "action": "classificar_pedido",                                                                               │
│   "args": {                                                                                                     │
│     "texto": "Solicito o total agregado por NCM e por mês, sem identificação dos contribuintes."                │
│   }                                                                                                             │
│ }                                                                                                               │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: Preciso classificar o pedido para identificar os campos relevantes e avaliar o risco de sigilo fiscal.

Action: classificar_pedido

Args: {"texto": "Solicito o total agregado por NCM e por mês, sem identificação dos contribuintes."}

...

──────────────────────────────────────────────────── Rodada 4 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ ```json                                                                                                         │
│ {                                                                                                               │
│   "thought": "A minuta está completa e atende aos requisitos, posso finalizar a resposta.",                     │
│   "action": "finish",                                                                                           │
│   "args": {                                                                                                     │
│     "decisao": "atendimento_parcial",                                                                           │
│     "resposta_final": "Solicitamos seu contato para fornecer o relatório agregado por NCM e por mês, sem        │
│ identificação dos contribuintes. A alternativa proposta preserva o recorte analítico principal e não apresenta  │
│ risco de exposição de dados sigilosos. Campos sugeridos: NCM, unidade fiscal, mês, valor total agregado. Campos │
│ excluídos: CNPJ, dados individualizados por contribuinte. Esta minuta não cita legislação real como fundamento  │
│ definitivo e deve ser revisada por responsável competente."                                                     │
│   }                                                                                                             │
│ }                                                                                                               │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: A minuta está completa e atende aos requisitos, posso finalizar a resposta.

╭──────────────────────────────────────────────────── Finish ─────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "decisao": "atendimento_parcial",                                                                             │
│   "resposta_final": "Solicitamos seu contato para fornecer o relatório agregado por NCM e por mês, sem          │
│ identificação dos contribuintes. A alternativa proposta preserva o recorte analítico principal e não apresenta  │
│ risco de exposição de dados sigilosos. Campos sugeridos: NCM, unidade fiscal, mês, valor total agregado. Campos │
│ excluídos: CNPJ, dados individualizados por contribuinte. Esta minuta não cita legislação real como fundamento  │
│ definitivo e deve ser revisada por responsável competente."                                                     │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'decisao': 'atendimento_parcial',
 'resposta_final': 'Solicitamos seu contato para fornecer o relatório agregado por NCM e por mês, sem identificação dos contribuintes. A alternativa proposta preserva o recorte analítico principal e não apresenta risco de exposição de dados sigilosos. Campos sugeridos: NCM, unidade fiscal, mês, valor total agregado. Campos excluídos: CNPJ, dados individualizados por contribuinte. Esta minuta não cita legislação real como fundamento definitivo e deve ser revisada por responsável competente.'}

In [ ]:
pedido_3 = "Solicito lista de empresas, CNPJ, valores e destinatários das operações."

resultado_3 = executar_agente_react(pedido_3)
resultado_3["decisao_final"]

## Exemplos com LLM

Os exemplos abaixo executam o ciclo ReAct completo com `executar_agente_react`: a LLM recebe o pedido, escolhe ferramentas, observa os retornos do Python e só então finaliza com a minuta.

In [ ]:
pedido_negativa = (
    "Solicito a lista completa de empresas, CNPJ, valores, remetentes e destinatários "
    "das operações fiscais de março de 2026. Não aceito dados agregados; preciso dos registros individualizados."
)

resultado_negativa = executar_agente_react(pedido_negativa)
resultado_negativa["decisao_final"]

In [42]:
pedido_atendimento_integral = (
    "Solicito apenas o total agregado de operações fiscais por NCM e por mês em 2026, "
    "sem CNPJ, sem empresas, sem remetentes, sem destinatários e sem qualquer identificação individualizada."
)

resultado_atendimento_integral = executar_agente_react(pedido_atendimento_integral)
resultado_atendimento_integral["decisao_final"]

──────────────────────────────────────────────────── Rodada 1 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ ```json                                                                                                         │
│ {                                                                                                               │
│   "thought": "Preciso classificar o pedido para identificar os campos relevantes e avaliar o risco de sigilo    │
│ fiscal.",                                                                                                       │
│   "action": "classificar_pedido",                                                                               │
│   "args": {                                                                                                     │
│     "texto": "Solicito apenas o total agregado de operações fiscais por NCM e por mês em 2026, sem CNPJ, sem    │
│ empresas, sem remetentes, sem destinatários e sem qualquer identificação individualizada."                      │
│   }                                                                                                             │
│ }                                                                                                               │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: Preciso classificar o pedido para identificar os campos relevantes e avaliar o risco de sigilo fiscal.

Action: classificar_pedido

Args: {"texto": "Solicito apenas o total agregado de operações fiscais por NCM e por mês em 2026, sem CNPJ, sem 
empresas, sem remetentes, sem destinatários e sem qualquer identificação individualizada."}

...

──────────────────────────────────────────────────── Rodada 3 ─────────────────────────────────────────────────────

╭───────────────────────────────────────────── Resposta bruta da LLM ─────────────────────────────────────────────╮
│ ```json                                                                                                         │
│ {                                                                                                               │
│   "thought": "A minuta de resposta está de acordo com a análise do pedido e a decisão de negar o acesso. Agora, │
│ finalizo o processo.",                                                                                          │
│   "action": "finish",                                                                                           │
│   "args": {                                                                                                     │
│     "decisao": "negativa",                                                                                      │
│     "resposta_final": "Negamos o fornecimento do relatório solicitado. O pedido contém informações que          │
│ representam alto risco de sigilo fiscal, incluindo CNPJ, destinatários e remetentes. Não é possível fornecer    │
│ dados agregados para atender à solicitação. Esta decisão está em consonância com as normas de proteção de dados │
│ e a legislação vigente."                                                                                        │
│   }                                                                                                             │
│ }                                                                                                               │
│ ```                                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thought: A minuta de resposta está de acordo com a análise do pedido e a decisão de negar o acesso. Agora, finalizo
o processo.

╭──────────────────────────────────────────────────── Finish ─────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "decisao": "negativa",                                                                                        │
│   "resposta_final": "Negamos o fornecimento do relatório solicitado. O pedido contém informações que            │
│ representam alto risco de sigilo fiscal, incluindo CNPJ, destinatários e remetentes. Não é possível fornecer    │
│ dados agregados para atender à solicitação. Esta decisão está em consonância com as normas de proteção de dados │
│ e a legislação vigente."                                                                                        │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'decisao': 'negativa',
 'resposta_final': 'Negamos o fornecimento do relatório solicitado. O pedido contém informações que representam alto risco de sigilo fiscal, incluindo CNPJ, destinatários e remetentes. Não é possível fornecer dados agregados para atender à solicitação. Esta decisão está em consonância com as normas de proteção de dados e a legislação vigente.'}

## Como a LLM responde passo a passo

1. A LLM recebe o pedido e o prompt de sistema.
2. Ela escolhe uma ação em JSON.
3. O código Python valida a ação.
4. O código Python executa a ferramenta.
5. A ferramenta retorna uma `Observation`.
6. A `Observation` é adicionada ao histórico.
7. A LLM usa a `Observation` para decidir a próxima ação.
8. O ciclo continua até `Finish`.

Fluxo textual:

Pedido do usuário  
↓  
LLM escolhe Action  
↓  
Python executa ferramenta  
↓  
Python gera Observation  
↓  
Observation volta ao histórico  
↓  
LLM decide próxima etapa  
↓  
Finish

## Quando usar ReAct?

Use ReAct quando a LLM precisa interagir com uma fonte externa, ferramenta ou ambiente antes de responder.

Exemplos:

- consulta a documentos;
- análise de pedido de acesso à informação;
- verificação de disponibilidade de relatório;
- auditoria;
- fiscalização;
- diagnóstico técnico;
- atendimento com APIs;
- agentes que precisam consultar bases externas;
- processos nos quais a resposta depende de evidências intermediárias.

O padrão é especialmente útil quando a resposta final deve depender de evidências obtidas passo a passo.

## Quando não usar ReAct?

ReAct tende a ser desnecessário quando a tarefa pode ser resolvida diretamente com o contexto já disponível.

Exemplos:

- explicações simples;
- reescrita de texto;
- tradução;
- perguntas conceituais diretas;
- tarefas que não exigem ferramentas;
- respostas que podem ser dadas diretamente com base no contexto fornecido.

Nesses casos, ReAct pode deixar o processo mais lento e mais complexo do que precisa ser.

## Cuidados importantes

- Não expor chave de API.
- Usar `OLLAMA_API_KEY` por variável de ambiente ou `getpass`, nunca colada em célula do notebook.
- Usar `ollama signin` apenas no ambiente local do Windows quando o modo for `cloud_proxy_signin`.
- Não enviar dados reais ou sensíveis quando `MODO_OLLAMA` usar cloud.
- Validar ações antes de executar ferramentas.
- Limitar número de rodadas.
- Não deixar a LLM inventar observações.
- Usar JSON estruturado.
- Registrar logs das rodadas.
- Separar decisão da LLM e execução da ferramenta.
- Evitar expor raciocínio interno longo.
- Usar ferramentas determinísticas quando o objetivo for ensino e depuração.
- Tratar respostas fora do formato esperado.
- Nunca usar `eval` para executar ações escolhidas pela LLM.
- Deixar claro quando o exemplo for fictício.
- Lembrar que modelos pequenos, como `gemma3:4b`, podem exigir prompts mais rígidos.
- Lembrar que modelos cloud dependem de chave ou login, internet e limites do plano Free.

## Diferença entre ReAct real e simulação

Simulação ruim:

A LLM escreve, em uma única resposta:

`Thought → Action → Observation → Finish`

Problema: a `Observation` foi inventada pela própria LLM.

ReAct real:

A LLM escreve:

`Thought → Action`

O sistema executa a ação.

O sistema devolve:

`Observation`

A LLM continua a partir dessa observação.

Essa separação é o núcleo do ReAct: a LLM decide a próxima ação, mas quem executa a ferramenta e produz a observação é o código.

## Resumo

- ReAct combina raciocínio orientado e uso de ferramentas.
- O modelo não deve inventar observações.
- O código é responsável por executar ferramentas.
- A LLM decide o próximo passo com base nas observações.
- O padrão é útil para tarefas de triagem, auditoria, atendimento, diagnóstico e análise com consulta a bases.
- Neste notebook, o exemplo é fictício e pode usar Ollama local com `gemma3:4b`, proxy local com `gemma4:31b-cloud` ou API cloud direta com `gemma4:31b`.
- Se o modelo cloud falhar por limite, autenticação ou indisponibilidade, o notebook faz fallback para `gemma3:4b`.